# Lesson 4.3 — Math and Statistics with Arrays

**Objectives**
- Do element-wise math between arrays and understand simple broadcasting
- Summarise an array with `sum`, `mean`, `min`, `max`, `std` and pick an `axis`
- Use `np.where` for if/else on a whole array
- Connect NumPy back to pandas using the course CO2 dataset

In [ ]:
import numpy as np

## 1. Element-wise math

When two arrays have the same shape, `+ - * /` work item by item. No loop needed:

In [ ]:
prices = np.array([10.0, 20.0, 30.0])
quantities = np.array([2, 5, 1])

print(prices * quantities)         # revenue per product
print((prices * quantities).sum()) # total revenue

NumPy also has vectorised versions of the math functions you know, e.g. `np.sqrt`, `np.round`, `np.abs`, `np.log`, `np.exp`:

In [ ]:
values = np.array([1, 4, 9, 16])

print(np.sqrt(values))
print(np.log(values))
print(np.round(np.sqrt([2, 3, 5]), 2))

## 2. Broadcasting (the simple version)

**Broadcasting** is NumPy's rule for combining arrays of *different* shapes. The simplest and
most common case: a single number is "stretched" to match the array. You already used this
with `arr * 2` in Lesson 4.1.

The next most common case: a 1D array is stretched across each row of a 2D array, as long as
the lengths match.

In [ ]:
table = np.array(
    [
        [1, 2, 3],
        [4, 5, 6],
    ]
)

print(table * 10)                         # a number is broadcast to every item
print(table + np.array([100, 200, 300]))  # a 1D row is broadcast to every row

If the shapes cannot be lined up, NumPy raises an error rather than guessing:

In [ ]:
try:
    table + np.array([1, 2])
except ValueError as e:
    print("Error:", e)

## 3. Aggregations: `sum`, `mean`, `min`, `max`, `std`

These *reduce* an array down to one number. They are available both as methods (`arr.mean()`) and functions (`np.mean(arr)`):

In [ ]:
temps = np.array([18.2, 25.6, 30.1, 22.8, 27.4, 15.9])

print("sum :", temps.sum())
print("mean:", temps.mean())
print("min :", temps.min())
print("max :", temps.max())
print("std :", temps.std())       # standard deviation: how spread out the values are
print("median:", np.median(temps))

`argmin` / `argmax` give the **position** of the smallest / largest value rather than the value itself:

In [ ]:
print(temps.argmax())          # position of the hottest day
print(temps[temps.argmax()])   # the hottest value

## 4. Aggregating along an `axis`

On a 2D array you choose which direction to collapse with `axis`:

- `axis=0` — collapse the rows, giving one result **per column**
- `axis=1` — collapse the columns, giving one result **per row**
- no `axis` — collapse everything to a single number

A handy way to remember: the axis you name is the one that *disappears*.

In [ ]:
# 3 stores (rows) x 4 months (columns)
sales = np.array(
    [
        [120, 135, 150, 160],
        [80, 95, 70, 110],
        [200, 210, 190, 220],
    ]
)

print("total per month (axis=0):", sales.sum(axis=0))
print("total per store (axis=1):", sales.sum(axis=1))
print("grand total            :", sales.sum())

In [ ]:
print("best month per store:", sales.argmax(axis=1))
print("mean per month      :", sales.mean(axis=0).round(1))

## 5. `np.where`: if / else for a whole array

`np.where(condition, value_if_true, value_if_false)` builds a new array by choosing from two options for every item:

In [ ]:
temps = np.array([18.2, 25.6, 30.1, 22.8, 27.4, 15.9])

labels = np.where(temps > 25, "hot", "mild")
print(labels)

In [ ]:
# the two options can be arrays too: give a 10% discount only where sales were low
discounted = np.where(sales < 100, sales * 0.9, sales)
print(discounted)

## 6. Sorting and unique values

In [ ]:
scores = np.array([88, 92, 75, 64, 95, 81, 75])

print(np.sort(scores))          # returns a sorted copy
print(np.argsort(scores))       # positions that would sort the array
print(np.unique(scores))        # distinct values, sorted

In [ ]:
values, counts = np.unique(scores, return_counts=True)
print(values)
print(counts)

## 7. NumPy and pandas together

Every pandas column is backed by a NumPy array. `.to_numpy()` hands you that array, and pandas
methods like `.mean()` call NumPy for you. Let's look at the course CO2 dataset from Module 3.

In [ ]:
import pandas as pd

co2_emission = pd.read_csv("../../data/owid-co2-data.csv", sep=",")
canada = co2_emission.loc[co2_emission["country"] == "Canada", ["year", "co2", "co2_per_capita"]]
canada.head()

In [ ]:
co2_values = canada["co2"].to_numpy()

print(type(co2_values))
print(co2_values.dtype)
print(co2_values.shape)

Real data has missing values, which NumPy stores as `nan` ("not a number"). Canada's `co2_per_capita`
column is missing for the earliest years. A plain `.mean()` becomes `nan` if **any** value is missing,
so use the `nan`-aware versions (`np.nanmean`, `np.nanmax`, `np.nanargmax`, ...) that skip them:

In [ ]:
per_capita = canada["co2_per_capita"].to_numpy()

print("first values  :", per_capita[:5])
print("missing count :", np.isnan(per_capita).sum())
print("plain mean    :", per_capita.mean())
print("nan-aware mean:", np.nanmean(per_capita).round(2))
print("nan-aware max :", np.nanmax(per_capita))

Now the NumPy skills from this module apply directly. Which year had Canada's highest emissions?

In [ ]:
years = canada["year"].to_numpy()

position_of_max = co2_values.argmax()
print("year:", years[position_of_max], " co2:", co2_values[position_of_max])

And the same `np.where` idea works as a new pandas column:

In [ ]:
canada = canada.copy()
canada["level"] = np.where(canada["co2"] > 500, "high", "low")
canada.tail()

## Try it yourself

Use the `sales` array (3 stores x 4 months) from Section 4.

1. Compute the average sales **per store** (one number per row)
2. Find which month (column position) had the highest total sales across all stores
3. Build a new array with the label `"good"` where sales are at least 150 and `"low"` otherwise
4. Using the CO2 data: select `United States` rows, convert the `co2` column to a NumPy array, and print its nan-aware mean rounded to 2 decimals

In [ ]:
sales = np.array(
    [
        [120, 135, 150, 160],
        [80, 95, 70, 110],
        [200, 210, 190, 220],
    ]
)

# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO

### Solution

In [ ]:
# 1.
print(sales.mean(axis=1))

In [ ]:
# 2.
print(sales.sum(axis=0).argmax())

In [ ]:
# 3.
print(np.where(sales >= 150, "good", "low"))

In [ ]:
# 4.
us_co2 = co2_emission.loc[co2_emission["country"] == "United States", "co2"].to_numpy()
print(np.nanmean(us_co2).round(2))